In [13]:
import os
%pwd

'/mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/NLP/End-to-End-NLP-Text-Summarization-with-Hugging-Face-Transformers'

In [14]:
os.chdir("../")

In [15]:
%pwd

'/mnt/D6A6D17EA6D15F93/AI-ML/REAL WORLD PROJECTS/NLP'

In [17]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    eval_strategy: str       # updated from evaluation_strategy
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int



In [5]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config=self.config.model_trainer
        params=self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config=ModelTrainerConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_ckpt = config.model_ckpt,
            num_train_epochs = params.num_train_epochs,
            warmup_steps = params.warmup_steps,
            per_device_train_batch_size = params.per_device_train_batch_size,
            weight_decay = params.weight_decay,
            logging_steps = params.logging_steps,
            eval_strategy = params.eval_strategy,
            eval_steps = params.eval_steps,
            save_steps = params.save_steps,
            gradient_accumulation_steps = params.gradient_accumulation_steps
        )
        return model_trainer_config



In [7]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

In [11]:

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):

        # -----------------------------
        # CUDA memory configuration
        # -----------------------------
        os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        device = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"Using device: {device}")

        # -----------------------------
        # Load tokenizer
        # -----------------------------
        tokenizer = AutoTokenizer.from_pretrained(
            self.config.model_ckpt
        )

        # -----------------------------
        # Load Pegasus model
        # -----------------------------
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_ckpt,
            torch_dtype=(
                torch.float16
                if device == "cuda"
                else torch.float32
            )
        ).to(device)

        # -----------------------------
        # Data collator
        # -----------------------------
        seq2seq_data_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=model_pegasus
        )

        # -----------------------------
        # Load processed dataset
        # -----------------------------
        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        print("Dataset loaded successfully")
        print(dataset_samsum_pt)

        # -----------------------------
        # Training arguments
        # -----------------------------
        trainer_args = TrainingArguments(
            output_dir=str(self.config.root_dir),

            num_train_epochs=self.config.num_train_epochs,

            warmup_steps=self.config.warmup_steps,

            per_device_train_batch_size=(
                self.config.per_device_train_batch_size
            ),

            per_device_eval_batch_size=(
                self.config.per_device_train_batch_size
            ),

            weight_decay=self.config.weight_decay,

            logging_steps=self.config.logging_steps,

            # IMPORTANT:
            # New Transformers versions use eval_strategy
            eval_strategy=self.config.eval_strategy,

            eval_steps=self.config.eval_steps,

            save_steps=int(float(self.config.save_steps)),

            gradient_accumulation_steps=(
                self.config.gradient_accumulation_steps
            ),

            # Enable FP16 only when CUDA is available
            fp16=(device == "cuda"),

            optim="adafactor",

            gradient_checkpointing=True,

            report_to="none"
        )

        # -----------------------------
        # Trainer
        # -----------------------------
        trainer = Trainer(
            model=model_pegasus,

            args=trainer_args,

            processing_class=tokenizer,

            data_collator=seq2seq_data_collator,

            train_dataset=dataset_samsum_pt["test"],

            eval_dataset=dataset_samsum_pt["validation"]
        )

        # -----------------------------
        # Start training
        # -----------------------------
        print("Starting training...")

        trainer.train()

        # -----------------------------
        # Save model
        # -----------------------------
        model_output_path = os.path.join(
            self.config.root_dir,
            "pegasus-samsum-model"
        )

        tokenizer_output_path = os.path.join(
            self.config.root_dir,
            "tokenizer"
        )

        model_pegasus.save_pretrained(
            model_output_path
        )

        tokenizer.save_pretrained(
            tokenizer_output_path
        )

        print("Model saved successfully!")
        print(f"Model path: {model_output_path}")
        print(f"Tokenizer path: {tokenizer_output_path}")



In [ ]:
config = ConfigurationManager()
model_trainer_config = config.get_model_trainer_config()
model_trainer = ModelTrainer(config=model_trainer_config)
model_trainer.train()